[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/36_explainable_ai/03_pitfalls_of_explanations.ipynb)

# Bài 03 — Cạm bẫy của giải thích mô hình

**🎯 Mục tiêu**
- **Đặc trưng tương quan**: độ quan trọng bị chia sẻ/thay thế; các định nghĩa "quan trọng" khác nhau trả lời câu hỏi khác nhau.
- **Hiệu ứng Rashomon**: nhiều mô hình tốt ngang nhau, giải thích khác nhau → giải thích nói về **mô hình**, không về **thế giới**.
- Dùng giải thích để **phát hiện rò rỉ** dữ liệu.
- Giải thích phụ thuộc **điểm tham chiếu** (dữ liệu nền).
- **Công bằng**: bỏ thuộc tính nhạy cảm không loại bỏ phân biệt — biến ủy nhiệm (proxy).
- Kiểm tra **độ trung thực** của lời giải thích; danh sách kiểm tra khi dùng XAI.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/36_explainable_ai

In [ ]:
import itertools
from math import factorial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
raw = make_customer_churn(20000, seed=0)
X = pd.DataFrame({
    "tenure": raw.tenure_months, "monthly": raw.monthly_charges, "total": raw.total_charges,
    "support_calls": raw.support_calls, "month_to_month": (raw.contract == "month-to-month").astype(int),
    "two_year": (raw.contract == "two_year").astype(int), "fiber": (raw.internet_service == "fiber").astype(int),
    "ewallet": (raw.payment_method == "e-wallet").astype(int), "senior": raw.senior.astype(int),
    "n_products": raw.n_products,
})
y = raw.churn.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
FEATURES = list(X.columns)


def fit(columns, X_tr=X_train):
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(X_tr[columns], y_train)


def perm_importance(model, columns, X_te=X_test):
    r = permutation_importance(model, X_te[columns], y_test, scoring="roc_auc", n_repeats=5, random_state=0)
    return pd.Series(r.importances_mean, index=columns)

## 1. Đặc trưng tương quan: độ quan trọng bị chia và bị thay thế

`total` ≈ `tenure` × `monthly`. Ba mô hình boosting: dùng mọi đặc trưng; bỏ `tenure`; bỏ `total`.

In [ ]:
variants = {"đủ đặc trưng": FEATURES, "bỏ tenure": [c for c in FEATURES if c != "tenure"], "bỏ total": [c for c in FEATURES if c != "total"]}
rows = {}
for name, cols in variants.items():
    m = fit(cols)
    auc = roc_auc_score(y_test, m.predict_proba(X_test[cols])[:, 1])
    pi = perm_importance(m, cols)
    rows[name] = {"ROC-AUC": auc, **{f"perm({c})": pi.get(c, np.nan) for c in ["tenure", "monthly", "total"]}}
print(pd.DataFrame(rows).T.round(4))

In [ ]:
importance = pd.DataFrame(rows).T[[f"perm({c})" for c in ["tenure", "monthly", "total"]]]
importance.columns = ["tenure", "monthly", "total"]
ax = importance.plot(kind="bar", rot=0, figsize=(9, 3.6))
ax.set_ylabel("permutation importance\n(ROC-AUC giảm khi xáo trộn)")
ax.set_title("Ba mô hình tốt ngang nhau, \"độ quan trọng\" khác hẳn")
plt.show()

Ba mô hình **tốt ngang nhau**, nhưng:
- Có đủ đặc trưng: `tenure` quan trọng, `total` gần 0.
- Bỏ `tenure`: `total` **thay thế** nó — độ quan trọng tăng hơn 10 lần. Mô hình tái tạo thông tin tenure từ total/monthly.

Nếu chỉ thấy mô hình thứ hai, ta có thể kết luận "tổng cước tích luỹ là yếu tố then chốt của việc rời bỏ" — trong khi theo logit thật, `total` **không** có tác động trực tiếp.

### Mỗi định nghĩa "quan trọng" trả lời một câu hỏi khác

In [ ]:
full_model = fit(FEATURES)
base_auc = roc_auc_score(y_test, full_model.predict_proba(X_test)[:, 1])
drop_column = {}
for c in ["tenure", "total", "monthly", "month_to_month"]:
    cols = [x for x in FEATURES if x != c]
    drop_column[c] = base_auc - roc_auc_score(y_test, fit(cols).predict_proba(X_test[cols])[:, 1])
print(pd.DataFrame({"permutation": perm_importance(full_model, FEATURES)[list(drop_column)], "drop-column (train lại)": pd.Series(drop_column)}).round(4))

- **Permutation**: "mô hình **này** mất bao nhiêu nếu đặc trưng bị phá hỏng?" — `tenure` quan trọng vì mô hình đang dựa vào nó.
- **Drop-column** (train lại không có đặc trưng): "dữ liệu mất bao nhiêu **thông tin không thay thế được** nếu không có đặc trưng?" — `tenure` gần 0 vì `total` + `monthly` bù lại được; chỉ `month_to_month` thật sự không thay thế được.

Không định nghĩa nào sai; sai là dùng một con số mà không nói rõ nó trả lời câu hỏi nào.

## 2. Hiệu ứng Rashomon

Hai mô hình "bỏ tenure" và "bỏ total" có hiệu năng gần như bằng nhau nhưng kể hai câu chuyện khác nhau. Với một bài toán thường tồn tại **cả một tập** mô hình gần tối ưu (Rashomon set — Breiman, 2001; Fisher et al., 2019); giải thích của **một** mô hình chỉ là một câu chuyện trong số đó. Hệ quả:
- Giải thích mô tả **mô hình**, không phải **quan hệ nhân quả** trong thế giới — muốn biết "tăng X có làm giảm rời bỏ không" phải dùng thí nghiệm hoặc suy luận nhân quả (chương 38).
- Muốn kết luận về dữ liệu, xem độ quan trọng **qua nhiều mô hình tốt** (model class reliance), không chỉ một.

## 3. Dùng giải thích để phát hiện rò rỉ

Giả sử bảng dữ liệu có cột `retention_call` — nhân viên gọi giữ chân khách hàng. Nghe hợp lý, nhưng cuộc gọi đó diễn ra **sau khi** khách hàng báo muốn huỷ (60% người rời bỏ có cuộc gọi, chỉ 3% người ở lại).

In [ ]:
rng = np.random.default_rng(1)
call_all = np.where(np.r_[y_train, y_test] == 1, rng.random(len(X)) < 0.6, rng.random(len(X)) < 0.03).astype(int)
X_leak_train = X_train.assign(retention_call=call_all[:len(X_train)])
X_leak_test = X_test.assign(retention_call=call_all[len(X_train):])
leaky = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(X_leak_train, y_train)
print(f"ROC-AUC: không rò rỉ = {base_auc:.3f} | có retention_call = {roc_auc_score(y_test, leaky.predict_proba(X_leak_test)[:, 1]):.3f}")
pi_leak = permutation_importance(leaky, X_leak_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0)
print(pd.Series(pi_leak.importances_mean, index=X_leak_test.columns).sort_values(ascending=False).round(3).head(5))

AUC tăng vọt "đẹp như mơ", và permutation importance chỉ thẳng thủ phạm: một đặc trưng **áp đảo** mọi đặc trưng khác. Đây là cách dùng XAI có giá trị nhất trong thực tế: **gỡ lỗi**. Câu hỏi bắt buộc khi thấy một đặc trưng áp đảo: *giá trị này có sẵn **tại thời điểm dự đoán** không?* (chương 5 — data leakage).

## 4. Giải thích phụ thuộc điểm tham chiếu

Giá trị Shapley giải thích $f(x) - \mathbb E_{\text{nền}}[f]$: "khác **ai**?" phụ thuộc dữ liệu nền. Cùng một khách hàng, so với (a) toàn bộ khách hàng, (b) chỉ khách hàng hợp đồng 2 năm:

In [ ]:
def exact_shapley(f, x, background):
    d, cols, value = x.shape[1], list(x.columns), {}
    for size in range(d + 1):
        for S in itertools.combinations(range(d), size):
            Z = background.copy()
            for j in S:
                Z[cols[j]] = x[cols[j]].values[0]
            value[S] = f(Z).mean()
    phi = np.zeros(d)
    for j in range(d):
        for S, v_S in value.items():
            if j not in S:
                phi[j] += factorial(len(S)) * factorial(d - len(S) - 1) / factorial(d) * (value[tuple(sorted(S + (j,)))] - v_S)
    return pd.Series(phi, index=cols), value[()]


f = full_model.decision_function
customer = X_test[X_test.month_to_month == 1].iloc[[0]]
bg_all = X_train.sample(150, random_state=0)
bg_loyal = X_train[X_train.two_year == 1].sample(150, random_state=0)
phi_all, base_all = exact_shapley(f, customer, bg_all)
phi_loyal, base_loyal = exact_shapley(f, customer, bg_loyal)
print(f"log-odds của khách hàng = {f(customer)[0]:.2f} | nền 'mọi khách' E[f] = {base_all:.2f} | nền 'hợp đồng 2 năm' E[f] = {base_loyal:.2f}")
print(pd.DataFrame({"so với mọi khách": phi_all, "so với khách 2 năm": phi_loyal}).round(3).sort_values("so với khách 2 năm", key=abs, ascending=False).head(6))

Cùng một khách hàng, hai câu chuyện: so với **khách hàng trung bình**, khách này rủi ro **thấp hơn** — cước thấp và thời gian gắn bó kéo xuống, dù hợp đồng theo tháng đẩy lên. So với **khách hợp đồng 2 năm**, khách này lại rủi ro **cao hơn**, và `month_to_month` trở thành lý do chính (đóng góp gần gấp đôi); đóng góp của `tenure` còn **đổi dấu**. Không có lời giải thích "khách quan" tuyệt đối — luôn nói rõ **so với nhóm tham chiếu nào** (khách hàng trung bình? khách được duyệt? khách cùng phân khúc?).

## 5. Công bằng: bỏ thuộc tính nhạy cảm là chưa đủ

Dữ liệu tín dụng tổng hợp: nhóm A (30% dân số, thuộc tính nhạy cảm) sống tập trung ở một khu vực (80% vs 15%), thu nhập trung bình thấp hơn, và **quyết định duyệt trong quá khứ** (nhãn) đã thiên lệch thêm với nhóm A. Mô hình **không** dùng thuộc tính nhóm — chỉ dùng thu nhập và khu vực.

In [ ]:
rng = np.random.default_rng(5)
n = 20000
group_a = rng.random(n) < 0.3
district = np.where(group_a, rng.random(n) < 0.8, rng.random(n) < 0.15).astype(int)
income = rng.normal(50 - 8 * group_a, 10, n)
approved = (rng.random(n) < 1 / (1 + np.exp(-(0.1 * (income - 45) - 1.0 * group_a)))).astype(int)   # nhãn lịch sử thiên lệch
credit = pd.DataFrame({"income": income, "district": district})
C_tr, C_te, g_tr, g_te, a_tr, a_te = train_test_split(credit, group_a, approved, test_size=0.3, random_state=0)
for cols in [["income", "district"], ["income"]]:
    m = LogisticRegression().fit(C_tr[cols], a_tr)
    approve = m.predict_proba(C_te[cols])[:, 1] > 0.5
    print(f"đặc trưng {cols}: tỉ lệ duyệt nhóm A = {approve[g_te].mean():.1%} | nhóm khác = {approve[~g_te].mean():.1%} | "
          f"hệ số = {dict(zip(cols, m.coef_[0].round(2)))}")

- Mô hình chưa từng thấy thuộc tính nhóm vẫn duyệt nhóm A ít hơn **ba lần**: `district` là **biến ủy nhiệm** (proxy) — hệ số âm lớn của nó (giải thích!) lộ ra điều đó.
- Bỏ luôn `district`, chênh lệch vẫn lớn: thu nhập khác nhau thật giữa hai nhóm, và **nhãn huấn luyện** mang thiên lệch lịch sử — mô hình học lại thiên lệch đó.

Giải thích giúp **phát hiện** dấu hiệu (proxy), nhưng không chứng nhận công bằng. Đánh giá công bằng cần đo trực tiếp kết quả theo nhóm (tỉ lệ duyệt, tỉ lệ lỗi theo nhóm — demographic parity, equalized odds), xem lại cách tạo nhãn, và cân nhắc các định nghĩa công bằng (vốn có thể mâu thuẫn nhau). Thư viện: Fairlearn, AIF360.

## 6. Lời giải thích có trung thực không?

Một kiểm tra đơn giản (**deletion test**): nếu lời giải thích đúng, "xoá" các đặc trưng nó nói là quan trọng nhất (thay bằng giá trị trung bình) phải làm dự đoán thay đổi **nhiều hơn** xoá ngẫu nhiên cùng số đặc trưng.

In [ ]:
def deletion_effect(x, order, k, background_means):
    z = x.copy()
    for c in order[:k]:
        z[c] = background_means[c]
    return abs(f(x)[0] - f(z)[0])


means = X_train.mean()
sample = X_test.sample(40, random_state=3)
rng = np.random.default_rng(0)
by_shapley, by_random = {k: [] for k in [1, 2, 3]}, {k: [] for k in [1, 2, 3]}
for i in range(len(sample)):
    x = sample.iloc[[i]]
    phi_x, _ = exact_shapley(f, x, X_train.sample(50, random_state=i))
    order = list(phi_x.abs().sort_values(ascending=False).index)
    for k in [1, 2, 3]:
        by_shapley[k].append(deletion_effect(x, order, k, means))
        by_random[k].append(deletion_effect(x, list(rng.permutation(FEATURES)), k, means))
for k in [1, 2, 3]:
    print(f"xoá {k} đặc trưng: thay đổi |log-odds| TB — theo Shapley = {np.mean(by_shapley[k]):.3f} | ngẫu nhiên = {np.mean(by_random[k]):.3f}")

Xoá theo thứ tự Shapley làm dự đoán đổi nhiều hơn hẳn xoá ngẫu nhiên — dấu hiệu lời giải thích **trung thực với mô hình**. (Lưu ý: thay bằng trung bình cũng tạo tổ hợp có thể phi thực tế — mọi phép thử độ trung thực đều có giả định riêng.)

## 7. Danh sách kiểm tra khi dùng XAI

1. **Mục đích là gì?** Gỡ lỗi, giải thích cho người dùng cuối, kiểm toán, hay khám phá khoa học — mỗi mục đích cần phương pháp và mức độ chặt chẽ khác nhau.
2. **Thử mô hình dễ hiểu trước** — nếu tốt ngang hộp đen, khỏi cần giải thích hậu kiểm.
3. **Kiểm tra tương quan** giữa đặc trưng trước khi đọc importance/PDP; ưu tiên ALE, nhóm các đặc trưng tương quan.
4. **Nói rõ câu hỏi** mà con số trả lời (permutation vs drop-column vs Shapley) và **nhóm tham chiếu**.
5. **Ổn định**: nhiều seed, nhiều mẫu, nhiều mô hình tốt (Rashomon); báo cáo độ dao động.
6. **Không suy diễn nhân quả** từ giải thích mô hình.
7. **Kiểm tra độ trung thực** và cẩn thận với việc giải thích có thể bị thao túng (Slack et al., 2020: mô hình có thể được thiết kế để "qua mặt" LIME/SHAP).
8. **Con người**: lời giải thích phải hiểu được và hành động được với người nhận; đánh giá bằng nghiên cứu người dùng khi quan trọng.

## 🧠 Kiểm tra nhanh

1. Vì sao độ quan trọng của `total` thay đổi mạnh giữa các mô hình có cùng hiệu năng?
2. Permutation importance và drop-column importance trả lời hai câu hỏi nào?
3. Hiệu ứng Rashomon là gì? Hệ quả với việc diễn giải?
4. Dùng XAI để phát hiện rò rỉ như thế nào?
5. Vì sao giá trị Shapley phụ thuộc dữ liệu nền?
6. Vì sao bỏ thuộc tính nhạy cảm không đảm bảo công bằng?

## 📌 Tóm tắt

- Đặc trưng tương quan làm độ quan trọng bị chia/thay thế; các định nghĩa "quan trọng" trả lời câu hỏi khác nhau.
- Nhiều mô hình tốt ngang nhau có giải thích khác nhau: giải thích nói về mô hình, không phải nhân quả.
- XAI rất hữu ích để gỡ lỗi (rò rỉ, đường tắt, proxy); kết quả phụ thuộc nhóm tham chiếu; cần kiểm tra độ trung thực và ổn định.
- Công bằng phải đo trực tiếp theo nhóm; bỏ thuộc tính nhạy cảm không loại bỏ proxy và nhãn thiên lệch.

## 📚 Tài liệu tham khảo

- Breiman (2001) — *Statistical Modeling: The Two Cultures* (Rashomon effect)
- Fisher, Rudin & Dominici (2019) — *All Models are Wrong, but Many are Useful*
- Hooker, Mentch & Zhou (2021) — *Unrestricted permutation forces extrapolation*
- Slack et al. (2020) — *Fooling LIME and SHAP: Adversarial Attacks on Post hoc Explanation Methods*
- Barocas, Hardt & Narayanan — *Fairness and Machine Learning* (sách mở, [fairmlbook.org](https://fairmlbook.org))
- Kaur et al. (2020) — *Interpreting Interpretability: Understanding Data Scientists' Use of Interpretability Tools*